# Assignment 2

## Formalia:

Please read the [assignment overview page](https://github.com/suneman/socialdata2026/wiki/Assignments) carefully before proceeding. This page contains information about formatting (including formats etc.), group sizes, and many other aspects of handing in the assignment. 

_If you fail to follow these simple instructions, it will negatively impact your grade!_

**Due date and time**: 
 - The assignment is due on Monday April 6th, 2026 at 23:55. 
 - Hand via DTU Learn. 
 - You should simply hand in the link to the github page with your short data story. (You can't hand in a link directly, so just put the link to your website in a text file and upload that).

## A2: A short data story

This assignment is to create a short data-story based on the work we've done in class so far. In particular you will solve *Exercise 2.1* from *Week 8, Part 2*. The exercises for that week contain full details on how the story should be constructed.

You will need to think about how you place that page on your personal github page (there are several ways of doing that and it's not hard. It's OK to ask an LLM for help). At some point you'll also need to host your final project there, so keep that in mind too.

Working Title
Times When Data Lies: How Reporting and Summaries Distort SF Crime Narratives

Subtitle
What incident records reveal, what they hide, and how to read them responsibly. 
How reportung and summary choices can mislead interpretation of SF incident data.

1. Intro (short, 120-170 words)

Define the dataset in plain language: SF police incident reports, 2003-2025.
Clarify what a record means: a reported/recorded incident, not direct ground truth.
Thesis: data can mislead through measurement bias, observation bias, and aggregation bias.
Preview the 3 figures and what each contributes.

2. Section A: The Clock Looks Precise, But Isn’t (measurement + observation bias)

Lead claim: minute-level timestamps can look more precise than the underlying event timing.
Static figure:
Multi-panel static chart.
Panel 1: minute-of-hour heaping (spikes at round minutes).
Panel 2: hourly rhythm with smoothing to show robust pattern.
Caption goal:
Explain both what is visible and why precision is partially administrative.
Interpretation paragraph:
Add observation bias: policing and reporting intensity can shape when incidents are recorded.
Explicit caution: reported incidents are not equal to true prevalence.

3. Section B: The Average City Does Not Exist (aggregation bias)

Lead claim: city averages can hide sharply different district realities.
Map figure:
District map with normalized measure (ratio or per-capita proxy if available).
Highlight Tenderloin contrast and one counterexample district.
Caption goal:
Explain why normalized comparison is used instead of raw counts.
Interpretation paragraph:
Add focus-crime percentage contrast.
Explicit caution: district differences may include enforcement/reporting effects, not only behavior differences.

4. Section C: Better Reading Tools (hybrid author/reader-driven)

Lead claim: robust summaries reduce misinterpretation risk.
Interactive Plotly figure:
Default state: one clear comparison already visible.
Controls: switch between mean, median, and distribution view; district/category toggle.
Keep controls minimal so narrative remains guided.
Caption goal:
Show how conclusions change when moving from averages to robust summaries.
Interpretation paragraph:
Explain what remains stable across summary choices versus what flips.

5. Conclusion (120-180 words)

What we can say confidently:
Broad temporal rhythms and structural district differences are real in the reported data.
What we cannot claim from this dataset alone:
Direct causal claims about true prevalence changes.
Practical takeaway:
Responsible reading requires checking measurement quality, enforcement context, and summary choice.
End with a concise responsibility statement tied to Week 1 themes.
References (short list at end)

Segel and Heer (2010).
Richardson et al. (Week 1 reading).
SF data portal source docs.
One or two external contextual sources for policy/reporting changes.
Style Rules For Your Final Page

Keep each section to one central claim.
Use claim-first figure titles.
Use captions as interpretation, not only description.
Use same color meaning across all visuals.
Keep total length tight, around 700-1000 words.
If you want, next step I can draft the exact final text section-by-section in your voice, including final figure titles and caption wording you can paste directly into your page.

In [54]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

In [55]:
df = pd.read_csv("data/crime_data_2003_2025.csv")
df["Datetime"] = pd.to_datetime(df["Datetime"], errors='coerce')

# List of focus crimes
focus_crimes = [
    'Larceny/Theft',
    'Other/Miscellaneous',
    'Non-Criminal',
    'Assault',
    'Vandalism',
    'Motor Vehicle Theft',
    'Burglary',
    'Warrants',
    'Drug Offense',
    'Suspicious Occ',
    'Robbery',
    'Fraud',
    'Missing Person'
]

# Unique color code for each crime category
color_category_mapping = {
    "Larceny/Theft": px.colors.qualitative.Plotly[0],
    "Other/Miscellaneous": px.colors.qualitative.Plotly[1],
    "Non-Criminal": px.colors.qualitative.Plotly[2],
    "Assault": px.colors.qualitative.Plotly[3],
    "Vandalism": px.colors.qualitative.Plotly[4],
    "Motor Vehicle Theft": px.colors.qualitative.Plotly[5],
    "Burglary": px.colors.qualitative.Plotly[6],
    "Warrants": px.colors.qualitative.Plotly[7],    
    "Drug Offense": px.colors.qualitative.Plotly[8],
    "Suspicious Occ": px.colors.qualitative.Plotly[9],
    "Robbery": px.colors.qualitative.Dark24[21],
    "Fraud": px.colors.qualitative.Dark24[22],
    "Missing Person": px.colors.qualitative.Dark24[15]
}

df["Color"] = df["Category"].map(color_category_mapping)
df = df[df["Year"] != 2026]

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2566096 entries, 0 to 2566095
Data columns (total 12 columns):
 #   Column           Dtype         
---  ------           -----         
 0   Row ID           int64         
 1   Incident ID      int64         
 2   Datetime         datetime64[us]
 3   Year             int64         
 4   Month            int64         
 5   Day of Week      int64         
 6   Hour             str           
 7   Police District  str           
 8   Category         str           
 9   Longitude        float64       
 10  Latitude         float64       
 11  Color            str           
dtypes: datetime64[us](1), float64(2), int64(5), str(4)
memory usage: 234.9 MB


### Section A: A broken clock is right twice a day

In [57]:
# Section A (combined): Panel 1A + 1B on the left, Panel 2 center-right

df_focus = df[df["Category"].isin(focus_crimes)].copy()
df_focus = df_focus.dropna(subset=["Datetime"])

df_focus["Minute"] = df_focus["Datetime"].dt.minute
minute_counts = df_focus["Minute"].value_counts().sort_index().reindex(range(60), fill_value=0)

total_events = int(minute_counts.sum())
expected_per_minute = total_events / 60 if total_events > 0 else np.nan

# Heaping metrics
is_multiple_5 = np.array([(m % 5) == 0 for m in range(60)])
is_quarter_hour = np.array([m in [0, 15, 30, 45] for m in range(60)])

p5_observed = minute_counts[is_multiple_5].sum() / total_events if total_events > 0 else np.nan
p15_observed = minute_counts[is_quarter_hour].sum() / total_events if total_events > 0 else np.nan
h5 = p5_observed / (12 / 60) if total_events > 0 else np.nan
h15 = p15_observed / (4 / 60) if total_events > 0 else np.nan

ratio_to_uniform = minute_counts / expected_per_minute if total_events > 0 else pd.Series(np.nan, index=range(60))

# Hourly pattern
df_focus["Hour"] = df_focus["Datetime"].dt.hour
hour_counts = df_focus["Hour"].value_counts().sort_index().reindex(range(24), fill_value=0)
hour_share = hour_counts / hour_counts.sum() if hour_counts.sum() > 0 else hour_counts.astype(float)
hour_smooth = hour_share.rolling(window=3, center=True, min_periods=1).mean()

# ---- SUBPLOTS WITH FORMATTED TITLES ----
fig_a = make_subplots(
    rows=1, cols=2,
    column_widths=[0.45, 0.55],
    horizontal_spacing=0.05,  
    subplot_titles=(
        "<b>Panel 1.</b> Heaping reveals reporting bias at administratively convenient times<br>"
        "<span style='font-size:12px;'>Recorded minutes cluster at round values rather than reflecting true event timing</span>",

        "<b>Panel 2.</b> Aggregated hourly patterns appear stable despite underlying reporting distortions<br>"
        "<span style='font-size:12px;'>Smoothing highlights persistent rhythms but does not remove measurement bias</span>"
    )
)

# ---- FIX TITLE POSITION + SIZE ----
fig_a.update_annotations(
    font=dict(size=16),
    yshift=12  # move titles higher
)

# ---- PANEL 1: HEAPING ----
dynamic_colors = ["#E74C3C" if r > 1 else "#F5B7B1" for r in ratio_to_uniform.values]

text_labels = [f"{r:.2f}×" if r > 1 else "" for r in ratio_to_uniform.values]

fig_a.add_trace(
    go.Bar(
        x=list(range(60)),
        y=ratio_to_uniform.values,
        marker_color=dynamic_colors,
        opacity=0.8,
        text=text_labels,
        textposition="outside",
        textfont=dict(size=12, color="#333"),  
        textangle=0,
        texttemplate="%{text}",  
        constraintext="none",    
        name="Heaping factor",
        cliponaxis=False,
        hovertemplate="Minute %{x}<br>Observed / Expected: %{y:.2f}×<extra></extra>"
    ),
    row=1, col=1
)

fig_a.add_hline(y=1, line_width=3, line_dash="dash", line_color="#2C3E50", row=1, col=1)

# ---- PANEL 2: HOURLY PATTERN ----
fig_a.add_trace(
    go.Bar(
        x=list(range(24)),
        y=hour_share.values * 100,
        name="Observed hourly share",
        marker_color="#1E5BA8",
        opacity=0.55,
        hovertemplate="Hour %{x}<br>Share: %{y:.4f}%<extra></extra>"
    ),
    row=1, col=2
)

fig_a.add_trace(
    go.Scatter(
        x=list(range(24)),
        y=hour_smooth.values * 100,
        mode="lines+markers",
        name="3-hour smoothed rhythm",
        line=dict(color="#2C3E50", width=2.5),
        marker=dict(size=7),
        hovertemplate="Hour %{x}<br>Smoothed: %{y:.4f}%<extra></extra>"
    ),
    row=1, col=2
)

# ---- ANNOTATION PANEL 1 (REFRAMED: REPORTING BIAS) ----
annotation_text = (
    "<b>Heaping bias</b><br>"
    "Times cluster at :00, :15, :30, :45<br>"
    "<i>Recorded ≠ actual time</i>"
)

fig_a.add_annotation(
    x=29.7,
    y=ratio_to_uniform[30] - 0.7,
    text=annotation_text,
    showarrow=True,
    arrowhead=1,
    arrowsize=1,
    arrowwidth=2,
    ax=-120,
    ay=-100,
    align="left",
    xanchor="center",
    yanchor="bottom",
    # Box styling (clean + publication look)
    # bgcolor="rgba(255,255,255,0.95)",
    # bordercolor="#2C3E50",
    # borderwidth=1,
    # borderpad=6,
    font=dict(size=14, family="Arial", color="#2C3E50"),
    row=1, col=1
)

# ---- ANNOTATION PANEL 2 (THICKER ARROW) ----
annotation_text_panel_2 = (
    "<b>Aggregation hides bias</b><br>"
    "Smooth hourly pattern masks distorted timestamps<br>"
    "<i>Midnight wraps</i>"
)

fig_a.add_annotation(
    x=0,
    y=hour_share.values[0] * 100,
    text=annotation_text_panel_2,
    showarrow=True,
    arrowhead=1,
    arrowwidth=2,  # ✅ THICKER ARROW
    arrowsize=1,
    ax=50,
    ay=-40,
    align="left",
    yanchor="middle",
    xanchor="left",
    # Box styling (clean + publication look)
    # bgcolor="rgba(255,255,255,0.95)",
    # bordercolor="#2C3E50",
    # borderwidth=1,
    # borderpad=6,
    font=dict(size=14, family="Arial", color="#2C3E50"),
    row=1, col=2
)

# ---- AXES ----
fig_a.update_xaxes(
    tickmode="array",
    tickvals=list(range(0, 60, 10)) + [15, 1, 45],
    title_text="Minute of hour",
    row=1, col=1
)

fig_a.update_xaxes(
    tickmode="array",
    tickvals=list(range(24)),
    ticktext=[f"{h:02d}" for h in range(24)],
    title_text="Hour of day",
    row=1, col=2
)

fig_a.update_yaxes(
    title_text="Observed / Expected",
    row=1, col=1
)

fig_a.update_yaxes(
    title_text="Share of reported incidents (%)",
    row=1, col=2
)

# ---- LAYOUT ----
fig_a.update_layout(
    template="plotly_white",
    height=700,
    bargap=0.10,
    hovermode="x unified",

    hoverlabel=dict(
        font_size=12,
        font_family="Arial",
        bgcolor="white"
    ),

    legend=dict(
        orientation="h",
        y=-0.18,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=13),
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.1)",
        borderwidth=1
    ),

    margin=dict(l=70, r=60, t=110, b=100),
    width=1700,
)

fig_a.show()
# fig_a.write_html("section_a_reporting_bias_and_temporal_patterns.html", config={"responsive": True})
fig_a.write_image("section_a_reporting_bias_and_temporal_patterns.png", width=1700, height=700, scale=2)

**Section A — Heaping Reveals Measurement & Reporting Bias**

- **Claim:** The minute-of-hour heaping plot shows recorded times are frequently rounded, so minute-level timestamps in this dataset are not a reliable measure of true event timing.
- **What the plots show:** The heaping bar chart (minute-of-hour) exhibits strong spikes at :00, :30 and at 15‑minute increments (:15, :45), with smaller peaks at every 5‑minute mark. The hourly rhythm (smoothed hourly share) shows a stable daily pattern but does not remove the minute-level rounding visible in the heaping plot.
- **Why this happens:** Recording the exact moment of an incident is often impractical. Reporters (victims, officers, dispatchers, data-entry clerks) tend to round to convenient times, producing clustered “heaps.” This is an administrative/measurement artifact in the timestamp data, not necessarily a true concentration of events at those minutes.
- **Observation bias (working hours / policing intensity):** Many reports are shaped by when people interact with institutions. During business hours (roughly 09:00–18:00) categories like Fraud, Non‑Criminal, and Other/Miscellaneous concentrate around midday; evening/night categories (Assault, Robbery) shift later. These patterns reflect both when incidents occur and when they are observed or recorded (patrol schedules, call volumes, reporting behavior).
- **Midnight wrap and distribution diagnostics:** Standard linear boxplots treat time as 0–23 and mishandle events that straddle midnight (e.g., 23:00–01:00). That “wrap” artificially inflates variance and can pull medians toward the center. Violin/density plots better reveal true nocturnal bulges and avoid misleading wrap artifacts.
- **Explicit caution:** Reported incident counts are measures of recorded interactions with the system, not ground-truth prevalence. Differences between districts or times can reflect enforcement and reporting intensity as much as underlying behavior.
- **Practical implication for modeling (time-series / ML):** If you train models on raw minute timestamps, the model can learn administrative rounding rather than real signals. Best practices:
  - Use coarser time bins (hourly or multi-hour) or circular time features (sin/cos transforms) to avoid overfitting to heaped minutes.
  - Consider de-biasing steps (smoothing, removing heaped minutes, or modeling rounding explicitly).
  - Validate on holdout data that preserves temporal structure; prefer robust summary metrics (median, trimmed mean) when features depend on noisy timestamps.
  - Beware: choosing too fine a timestamp discretization can amplify recording artifacts and produce biased predictions.

Caption (how to read the figures): the minute-of-hour heap chart highlights rounding artifacts to inspect first; use the smoothed hourly plot to see broader rhythms that persist after accounting for minute-level noise.

### Section B: 

In [ ]:
# AGGREGATION BIAS EXAMPLE 4: Heatmap + City Trend reveals which districts diverge

# Prepare df_focus_clean for Section B analysis
df_focus_clean = df_focus.copy()
df_focus_clean["Year"] = df_focus_clean["Datetime"].dt.year
df_focus_clean["Day"] = df_focus_clean["Datetime"].dt.date

crime_data = df_focus_clean[df_focus_clean["Category"] == "Drug Offense"].copy()

# Exclude "Out of SF"
crime_data = crime_data[crime_data["Police District"] != "Out Of Sf"].copy()

# Create normalized counts by year (percentage of city total)
district_year = crime_data.groupby(["Year", "Police District"]).size().reset_index(name="count")
normalized = district_year.copy()
year_totals = district_year.groupby("Year")["count"].transform("sum")
normalized["pct"] = (normalized["count"] / year_totals * 100).round(1)

# City total by year (left subplot)
city_total = crime_data.groupby("Year").size().reset_index(name="count")

# Calculate trendline for city total
z = np.polyfit(city_total["Year"], city_total["count"], 5)
p = np.poly1d(z)
trendline_y = p(city_total["Year"])

# Pivot for heatmap
heatmap_data = normalized.pivot(index="Police District", columns="Year", values="pct")

# Custom red color scale (matches your bar chart palette)
custom_colorscale = [
    [0.0, "#EBF5FB"],   # very light blue (equivalent to FDEDEC)
    [0.2, "#AED6F1"],
    [0.4, "#5DADE2"],
    [0.6, "#3498DB"],   # your main blue
    [0.8, "#2E86C1"],
    [1.0, "#1E5BA8"]    # your dark blue
]

# Create subplots with styled titles
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<span style='font-size:18px;'><b>Panel 1.</b> Total Drug Offenses Over Time</span><br>"
        "<span style='font-size:14px;'>City-Level Trend Masks District Variation</span>",

        "<span style='font-size:18px;'><b>Panel 2.</b> District Share of Total Drug Offenses (%)</span><br>"
        "<span style='font-size:14px;'>Reveals Diverging Local Trends</span>"
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}]],
    horizontal_spacing=0.08,
    column_widths=[0.45, 0.55]
)

# Proper annotation styling (fixes your original issue)
fig.update_annotations(
    font=dict(size=16, family="Arial"),
    yshift=10
)

# LEFT: Time series with subtle fill
fig.add_trace(
    go.Scatter(
        x=city_total["Year"],
        y=city_total["count"],
        mode="lines+markers",
        name="Total Drug Offenses",
        line=dict(color="#2C3E50", width=2, shape="spline"),
        marker=dict(size=7),
        fill="tozeroy",
        fillcolor="rgba(44, 62, 80, 0.15)",  # subtle area fill
        hovertemplate="Year: %{x}<br>Total: %{y}<extra></extra>",
    ),
    row=1, col=1
)

# Trendline (aligned with your palette)
fig.add_trace(
    go.Scatter(
        x=city_total["Year"],
        y=trendline_y,
        mode="lines",
        name="Trend",
        line=dict(color="#E74C3C", width=3, dash="dash"),
        hovertemplate="Year: %{x}<br>Trend: %{y:.0f}<extra></extra>"
    ),
    row=1, col=1
)

# RIGHT: Heatmap with custom palette
fig.add_trace(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale=custom_colorscale,
        text=heatmap_data.values,
        texttemplate="%{text:.1f}", 
        textfont={"size": 10},
        colorbar=dict(
            title="<b>% of City Total</b>",
            len=0.85,
            y=0.5,
            tickfont=dict(size=11)
        ),
        showscale=True
    ),
    row=1, col=2
)
# ---- ADD ANNOTATION (Panel 1 insight box with arrow) ----

# Pick a point near the end of the trendline
x_annot = city_total["Year"].iloc[-2]
y_annot = trendline_y[-2]

fig.add_annotation(
    x=x_annot,
    y=y_annot,
    xref="x1",   # refers to first subplot
    yref="y1",
    
    text=(
        "<b>Upward trend since 2021</b><br>"
        "Drug Offense cases are increasing citywide.<br>"
        "However, this is not true for all districts.<br>"
        "<i>This is aggregation bias.</i>"
    ),

    showarrow=True,
    arrowhead=2,
    arrowsize=1.2,
    arrowwidth=2,
    arrowcolor="#2C3E50",

    # Position of text box
    ax=-100,   # move box to the right
    ay=-150,  # move box slightly down

    align="left",
    
    # Box styling (clean + publication look)
    # bgcolor="rgba(255,255,255,0.95)",
    # bordercolor="#2C3E50",
    # borderwidth=1,
    # borderpad=6,

    font=dict(size=14, family="Arial", color="#2C3E50")
)

# Axis labels (cleaner typography)
fig.update_xaxes(title_text="Year", row=1, col=1, title_font=dict(size=13))
fig.update_xaxes(title_text="Year", row=1, col=2, title_font=dict(size=13))
fig.update_yaxes(title_text="# Reported Incidents", row=1, col=1, title_font=dict(size=13))
fig.update_yaxes(title_text="Police District", row=1, col=2, title_font=dict(size=13))

# Layout (publication style)
fig.update_layout(
    height=700,
    template="plotly_white",
    hovermode="x unified",

    # Clean hover labels
    hoverlabel=dict(
        font_size=12,
        font_family="Arial",
        bgcolor="white"
    ),

    # Improved legend
    legend=dict(
        orientation="h",
        y=-0.18,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=13),
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.1)",
        borderwidth=1
    ),

    margin=dict(l=70, r=100, t=120, b=110)
)

fig.show()

In [ ]:
# Section B: Aggregation Bias + Geographic Hotspots (Police District GeoJSON + Fixed SF Zoom)
import re
import json
from shapely.geometry import shape

# Prepare df_focus_clean for Section B analysis
df_focus_clean = df_focus.copy()
df_focus_clean["Year"] = df_focus_clean["Datetime"].dt.year
df_focus_clean["Day"] = df_focus_clean["Datetime"].dt.date

crime_data = df_focus_clean[df_focus_clean["Category"] == "Drug Offense"].copy()

# Exclude "Out of SF"
crime_data = crime_data[crime_data["Police District"] != "Out Of Sf"].copy()

# Create normalized counts by year (percentage of city total)
district_year = crime_data.groupby(["Year", "Police District"]).size().reset_index(name="count")
normalized = district_year.copy()
year_totals = district_year.groupby("Year")["count"].transform("sum")
normalized["pct"] = (normalized["count"] / year_totals * 100).round(1)

# City total by year (left subplot)
city_total = crime_data.groupby("Year").size().reset_index(name="count")

# Calculate trendline for city total
z = np.polyfit(city_total["Year"], city_total["count"], 5)
p = np.poly1d(z)
trendline_y = p(city_total["Year"])

# Pivot for heatmap
heatmap_data = normalized.pivot(index="Police District", columns="Year", values="pct")

# Shared blue scale used in panel 2
custom_colorscale = [
    [0.0, "#EBF5FB"],
    [0.2, "#AED6F1"],
    [0.4, "#5DADE2"],
    [0.6, "#3498DB"],
    [0.8, "#2E86C1"],
    [1.0, "#1E5BA8"]
]

# Custom density scale requested for panel 3
density_colorscale = [
    [0.0, "rgba(120, 120, 120, 0.18)"],
    [0.55, "#E74C3C"],
    [1.0, "#952115"]
]

# === CREATE SUBPLOTS: 2 ROWS ===
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.4, 0.6],
    subplot_titles=(
        "<span style='font-size:18px;'><b>Panel 1.</b> Total Drug Offenses Over Time<br><span style='font-size:14px;'>City-Level Trend Masks District Variation</span></span>",
        "<span style='font-size:18px;'><b>Panel 2.</b> District Share of Total Drug Offenses (%)<br><span style='font-size:14px;'>Reveals Diverging Local Trends</span></span>",
        "<span style='font-size:18px;'><b>Panel 3.</b> District Aggregation Still Hides Street-Level Hotspots (2025, Drug Offenses)<br><span style='font-size:14px;'>Even within large police districts, offenses cluster on specific streets</span></span>",
        ""
    ),
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"type": "map", "colspan": 2}, None]
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.18
)

fig.update_annotations(font=dict(size=14, family="Arial"), yshift=10)

# === PANEL 1: TIME SERIES ===
fig.add_trace(
    go.Scatter(
        x=city_total["Year"],
        y=city_total["count"],
        mode="lines+markers",
        name="Total Drug Offenses",
        line=dict(color="#2C3E50", width=2, shape="spline"),
        marker=dict(size=7),
        fill="tozeroy",
        fillcolor="rgba(44, 62, 80, 0.15)",
        hovertemplate="Year: %{x}<br>Total: %{y}<extra></extra>",
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=city_total["Year"],
        y=trendline_y,
        mode="lines",
        name="Trend",
        line=dict(color="#E74C3C", width=3, dash="dash"),
        hovertemplate="Year: %{x}<br>Trend: %{y:.0f}<extra></extra>",
        showlegend=False
    ),
    row=1, col=1
)

# === PANEL 2: HEATMAP ===
fig.add_trace(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale=custom_colorscale,
        text=heatmap_data.values,
        texttemplate="%{text:.1f}",
        textfont={"size": 10},
        colorbar=dict(
            title="<b>% of City Total</b>",
            len=0.32,
            y=0.84,
            tickfont=dict(size=10)
        ),
        showscale=True,
        name="Share (%)"
    ),
    row=1, col=2
)

# ---- ANNOTATION ----
x_annot = city_total["Year"].iloc[-2]
y_annot = trendline_y[-2]

fig.add_annotation(
    x=x_annot,
    y=y_annot,
    xref="x1",
    yref="y1",
    text=(
        "<b>Upward trend since 2021</b><br>"
        "Drug Offense cases are increasing citywide.<br>"
        "However, this is not true for all districts.<br>"
        "<i>This is aggregation bias.</i>"
    ),
    showarrow=True,
    arrowhead=2,
    arrowsize=1.2,
    arrowwidth=2,
    arrowcolor="#2C3E50",
    ax=-100,
    ay=-150,
    align="left",
    font=dict(size=12, family="Arial", color="#2C3E50")
)

# === PANEL 3: POLICE DISTRICTS (sfpd.txt) + DENSITY ===
# Filter to 2025 and valid coordinates
data_2025 = crime_data[crime_data["Year"] == 2025].dropna(subset=["Latitude", "Longitude"]).copy()

# Load exact police district boundaries from GeoJSON
with open("sfpd.txt", "r") as f:
    district_geojson = json.load(f)

district_features = district_geojson.get("features", [])
district_names = [feat.get("properties", {}).get("DISTRICT", "UNKNOWN") for feat in district_features]

# Subtle gray district fill
fig.add_trace(
    go.Choroplethmap(
        geojson=district_geojson,
        locations=district_names,
        z=np.ones(len(district_names)),
        featureidkey="properties.DISTRICT",
        colorscale=[
            [0.0, "rgba(120, 120, 120, 0.18)"],
            [1.0, "rgba(120, 120, 120, 0.18)"]
        ],
        marker_opacity=1.0,
        marker_line_width=0,
        showscale=False,
        name="Police district fill",
        hovertemplate="District: %{location}<extra></extra>",
        subplot="map"
    )
)

# Drug-offense density overlay (lon/lat) using requested color scale
fig.add_trace(
    go.Densitymap(
        lat=data_2025["Latitude"],
        lon=data_2025["Longitude"],
        radius=15,
        colorscale=density_colorscale,
        showscale=True,
        opacity=0.7,
        colorbar=dict(
            title="<b>Drug Offense Density</b>",
            len=0.35,
            y=0.2,
            tickfont=dict(size=10)
        ),
        hovertemplate="Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<extra></extra>",
        name="Density",
        subplot="map"
    )
)

# Extract district boundary lines for top-layer outlines
outline_lon, outline_lat = [], []
label_lon, label_lat, label_text = [], [], []

for feat in district_features:
    geom = feat.get("geometry", {})
    props = feat.get("properties", {})
    district_name = props.get("DISTRICT", "UNKNOWN")

    try:
        shp = shape(geom)
        rp = shp.representative_point()
        label_lon.append(rp.x)
        label_lat.append(rp.y)
        label_text.append(district_name.title())
    except Exception:
        pass

    gtype = geom.get("type")
    coords = geom.get("coordinates", [])

    if gtype == "Polygon":
        polygons = [coords]
    elif gtype == "MultiPolygon":
        polygons = coords
    else:
        polygons = []

    for polygon in polygons:
        for ring in polygon:
            for lon, lat in ring:
                outline_lon.append(lon)
                outline_lat.append(lat)
            outline_lon.append(None)
            outline_lat.append(None)

# Top-layer district outlines
fig.add_trace(
    go.Scattermap(
        lon=outline_lon,
        lat=outline_lat,
        mode="lines",
        line=dict(color="rgba(20, 20, 20, 0.95)", width=1.7),
        name="Police district boundaries",
        hoverinfo="skip",
        subplot="map",
        showlegend=True
    )
)

# District name labels from GeoJSON geometry
fig.add_trace(
    go.Scattermap(
        lon=label_lon,
        lat=label_lat,
        mode="text",
        text=label_text,
        textposition="middle center",
        textfont=dict(size=11, color="rgba(35, 35, 35, 0.95)"),
        name="District names",
        hoverinfo="skip",
        subplot="map",
        showlegend=False
    )
)

# === UPDATE AXES ===
fig.update_xaxes(title_text="Year", row=1, col=1, title_font=dict(size=12))
fig.update_xaxes(title_text="Year", row=1, col=2, title_font=dict(size=12))
fig.update_yaxes(title_text="# Reported Incidents", row=1, col=1, title_font=dict(size=12))
fig.update_yaxes(title_text="Police District", row=1, col=2, title_font=dict(size=12))

# === UPDATE LAYOUT ===
fig.update_layout(
    template="plotly_white",
    height=1200,
    hovermode="closest",
    margin=dict(l=70, r=100, t=120, b=150),
    width=1500,

    # Keep base map clean; district names are rendered by our text trace.
    map=dict(
        style="white-bg",
        center=dict(lat=37.7749, lon=-122.4194),
        zoom=11.7,
        bearing=0,
        pitch=0,
        uirevision="sf-fixed-view",
        layers=[
            dict(
                sourcetype="raster",
                source=[
                    "https://a.basemaps.cartocdn.com/rastertiles/light_nolabels/{z}/{x}/{y}.png",
                    "https://b.basemaps.cartocdn.com/rastertiles/light_nolabels/{z}/{x}/{y}.png",
                    "https://c.basemaps.cartocdn.com/rastertiles/light_nolabels/{z}/{x}/{y}.png"
                ],
                below="traces"
            )
        ]
    ),

    showlegend=True,
    legend=dict(
        orientation="h",
        y=-0.07,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=13),
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.1)",
        borderwidth=1
    ),
    hoverlabel=dict(
        font_size=11,
        font_family="Arial",
        bgcolor="white"
    )
)

fig.show()
fig.write_html("section_b_aggregation_bias_and_hotspots.html", config={"responsive": True})

**Section B — Aggregation Bias & Enforcement Effects**

- **Overview:** These panels analyze Drug Offenses (2003–2025) to show how a city-level story can hide sharply different local trajectories. We compare the city total and trend (Panel A), district shares over time (Panel B), and a street‑level density map (Panel C).

**Panel A — City total & trend**: The city-wide series and fitted trendline show the broad temporal pattern for Drug Offenses. The city aggregate looks comparatively steady for much of the series, but the trendline highlights an uptick since about 2021. That single-line view masks divergent district-level behavior shown in Panels B and C.

**Panel B — District share heatmap**: The heatmap displays each police district’s percent share of the city total by year. Some districts (notably Tenderloin and a couple of others) increase their share over time while many districts decline or remain low. The heatmap makes it clear that districts follow different trajectories even when the city total appears stable.

**Panel C — District map + street-level density**: The map overlays subtle district fills with a street-level density raster. Even inside large districts, incidents concentrate on particular streets and blocks — hotspots that district-level aggregates cannot show. These street clusters may reflect true concentration, enforcement focus, or reporting patterns.

**Interpretation (adjusted text for the notebook)**

The city-wide aggregate masks a critical truth: while San Francisco’s reported Drug Offenses appear relatively stable over 2003–2025 at the city level, individual districts tell very different stories. Tenderloin’s percentage share of the city total has risen, while many other neighborhoods have declined or diverged. Panel A’s trendline makes the city-level movement visible, but Panel B shows that this apparent stability comes from averaging very different local trends. Panel C demonstrates that even within a single district incidents cluster on specific streets, so district‑level summaries still hide important spatial concentration.

Raw counts alone cannot resolve whether observed changes reflect true incidence or changes in enforcement and reporting. A district that shows a rising share could be experiencing more incidents, more intensive policing, easier reporting, or some combination of those factors. To move from observed reports toward causal claims requires additional data on enforcement practice (patrol assignments, response volumes) and community reporting behavior.

**The key takeaway:** Aggregation flattens crucial variation. A 2× change in district share does not automatically mean twice the underlying criminal activity — it can mean twice the reporting or policing intensity. Treat city aggregates as a starting point, then disaggregate and map to understand where and why patterns differ.

**Suggested caption (short)**  
Left: city totals with trendline; center: district share heatmap (percent of city total) showing diverging local trends; bottom/map: street‑level density showing concentrated hotspots inside districts. Normalize before comparing districts; interpret differences with enforcement and reporting context in mind.

**Practical note for analysis and modelling**  
Always normalize by city totals or population when comparing districts and pair reported incidents with enforcement/response indicators. When building predictive models, use aggregated or circular time features and validate against holdouts — models trained on raw counts or fine-grained timestamps risk learning administrative artifacts (reporting/rounding) rather than substantive signals.

The Average City Does Not Exist. Reported crime in San Francisco shows one story at the city level—but when disaggregated by district, reveals another. When you average across neighborhoods, you hide how differently crime is distributed. Tenderloin and Southern Station report far more incidents than other districts, but normalized ratios show that enforcement resources concentrate in specific areas. A 2x ratio doesn't mean twice the criminal behavior—it means twice the reporting intensity, which reflects policing priorities, staffing levels, and possibly differing community-police engagement patterns. Cities are mosaics, not monoliths.

In [ ]:
# Section C: Interactive visualization (HTML-exportable)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _trimmed_mean(s, prop=0.1):
    if len(s) == 0:
        return np.nan
    ntrim = int(len(s) * prop)
    if ntrim == 0:
        return s.mean()
    s_sorted = np.sort(s)
    return s_sorted[ntrim:len(s_sorted) - ntrim].mean() if len(s_sorted) > 2 * ntrim else s.mean()



def _get_df():
    g = globals()
    for name in ("df_focus_clean", "df_focus", "df"):
        if name in g:
            return g[name].copy()
    raise RuntimeError("No dataframe found")


def create_section_c_plot(year=None, categories=None, normalize="ratio_to_city", min_days=10):
    """
    Create interactive Section C visualization.

    Parameters:
    - year: int or None (None = all years)
    - categories: list of category names or None (None = all)
    - normalize: "raw" or "ratio_to_city"
    - min_days: minimum sample size per district
    """
    df = _get_df()

    if "Datetime" in df.columns:
        df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
        df["Year"] = df["Datetime"].dt.year
        df["Day"] = df["Datetime"].dt.date

    df_sub = df.copy()
    if categories and len(categories) > 0:
        df_sub = df_sub[df_sub["Category"].isin(categories)]
    if year is not None:
        df_sub = df_sub[df_sub["Year"] == year]

    df_sub = df_sub[df["Police District"] != "Out Of Sf"]

    daily = df_sub.groupby(["Police District", "Day"]).size().reset_index(name="count")

    sample_counts = daily.groupby("Police District")["Day"].nunique()
    valid_districts = sample_counts[sample_counts >= min_days].index.tolist()
    daily = daily[daily["Police District"].isin(valid_districts)]

    if daily.empty:
        raise ValueError("No data after filtering")

    summaries = []
    for dname, g in daily.groupby("Police District"):
        arr = g["count"].values
        summaries.append(
            {
                "Police District": dname,
                "median": np.median(arr),
                "mean": np.mean(arr),
                "trimmed_mean": _trimmed_mean(arr, 0.1),
                "p25": np.percentile(arr, 25),
                "p75": np.percentile(arr, 75),
                "n_days": len(arr),
            }
        )
    sumdf = pd.DataFrame(summaries).set_index("Police District")

    fig = make_subplots(
        rows=1,
        cols=3,
        column_widths=[0.36, 0.32, 0.32],
        horizontal_spacing=0.09,
        subplot_titles=(
            "<span style='font-size:18px;'><b>Panel 1.</b> District Comparison Depends on Summary Metric</span>",
            "<span style='font-size:18px;'><b>Panel 2.</b> Distribution Reveals Daily Volatility</span>",
            "<span style='font-size:18px;'><b>Panel 3.</b> Mean-Median Gap Flags Skew</span>",
        ),
        specs=[[{}, {}, {}]],
    )

    metric_order = ["median", "mean", "trimmed_mean"]
    metric_label = {
        "median": "Median",
        "mean": "Mean",
        "trimmed_mean": "Trimmed Mean (10%)",
    }

    traces_panel_a = []
    for metric_name in metric_order:
        s = sumdf[metric_name].sort_values()

        if normalize == "ratio_to_city":
            city_ref = sumdf[metric_name].mean()
            s_plot = s / (city_ref if city_ref != 0 else 1.0)
            colors = ["#E74C3C" if v > 1.2 else "#3498DB" if v < 0.8 else "#95A5A6" for v in s_plot.values]
            x_title_panel_a = "Ratio to city average (=1.0)"
        else:
            s_plot = s
            colors = ["#1E5BA8"] * len(s_plot)
            x_title_panel_a = "Incidents per day"

        visible = metric_name == "median"
        fig.add_trace(
            go.Bar(
                x=s_plot.values,
                y=s_plot.index.tolist(),
                orientation="h",
                marker_color=colors,
                text=[f"{v:.2f}" for v in s_plot.values],
                textposition="outside",
                name=metric_label[metric_name],
                visible=visible,
                hovertemplate="%{y}<br>" + metric_label[metric_name] + ": %{x:.2f}<extra></extra>",
                opacity=0.7
            ),
            row=1,
            col=1,
        )
        traces_panel_a.append(len(fig.data) - 1)

    if normalize == "ratio_to_city":
        city_ref = sumdf["median"].mean()
        p25_plot = (sumdf["p25"] / city_ref).sort_values()
        p75_plot = (sumdf["p75"] / city_ref).sort_values()
    else:
        p25_plot = sumdf["p25"].sort_values()
        p75_plot = sumdf["p75"].sort_values()

    fig.add_trace(
        go.Bar(
            x=p75_plot.values,
            y=p75_plot.index.tolist(),
            orientation="h",
            name="75th percentile",
            marker_color="#1E5BA8",
            opacity=0.7,
            visible=visible,
            hovertemplate="%{y}<br>75th percentile: %{x:.2f}<extra></extra>",
            textposition="outside",

        ),
        row=1,
        col=1,
    )
    trace_p75 = len(fig.data) - 1

    fig.add_trace(
        go.Bar(
            x=p25_plot.values,
            y=p25_plot.index.tolist(),
            orientation="h",
            name="25th percentile",
            marker_color="#AED6F1",
            opacity=0.7,
            visible=False,
            hovertemplate="%{y}<br>25th percentile: %{x:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    trace_p25 = len(fig.data) - 1

    districts_top = sumdf.nlargest(3, "median").index.tolist()
    violin_palette = ["#2ECC71", "#16A085", "#72A99F"]

    for i, dname in enumerate(districts_top):
        df_dist = daily[daily["Police District"] == dname]
        if not df_dist.empty:
            fig.add_trace(
                go.Violin(
                    x=[dname] * len(df_dist),
                    y=df_dist["count"],
                    name=dname,
                    box_visible=True,
                    meanline_visible=True,
                    points=False,
                    fillcolor=violin_palette[i % len(violin_palette)],
                    line_color=violin_palette[i % len(violin_palette)],
                    opacity=0.55,
                    hovertemplate="District: " + dname + "<br>Incidents/day: %{y}<extra></extra>",
                    showlegend=False,
                ),
                row=1,
                col=2,
            )

    diff_df = sumdf.copy()
    diff_df["relative_diff_pct"] = (diff_df["mean"] - diff_df["median"]) / (
        diff_df["median"].replace(0, np.nan)
    ) * 100
    diff_df = diff_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["relative_diff_pct"]).sort_values("relative_diff_pct")

    fig.add_trace(
        go.Bar(
            x=diff_df["relative_diff_pct"].values,
            y=diff_df.index.tolist(),
            orientation="h",
            marker_color=["#EF9343" if v > 0 else "#1ABC9C" for v in diff_df["relative_diff_pct"].values],
            hovertemplate="%{y}<br>(Mean - Median)/Median: %{x:.1f}%<extra></extra>",
            showlegend=False,
            name="Mean-median gap",
            opacity=0.8
        ),
        row=1,
        col=3,
    )

    if normalize == "ratio_to_city":
        fig.add_vline(x=1.0, line_width=2, line_dash="dash", line_color="#2C3E50", row=1, col=1)

    fig.add_vline(x=0, line_width=2, line_dash="dot", line_color="#7F8C8D", row=1, col=3)

    if not diff_df.empty:
        top_skew = diff_df["relative_diff_pct"].idxmax()
        top_skew_val = diff_df["relative_diff_pct"].max()
        fig.add_annotation(
            x=top_skew_val,
            y=top_skew,
            xref="x3",
            yref="y3",
            text="<b>Largest skew</b><br>Mean pulled by spikes",
            showarrow=True,
            arrowhead=2,
            ax=80,
            ay=-25,
            font=dict(size=11, color="#2C3E50"),
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="rgba(44,62,80,0.25)",
            borderwidth=1,
        )

    metric_buttons = []
    total_traces = len(fig.data)

    subtitle_init = (
        f"<b>Metric:</b> {metric_label['median']} | "
        f"<b>Normalize:</b> {normalize} | "
        f"<b>Min days:</b> {min_days}"
    )

    fig.add_annotation(
        x=0.13,  # positioned to the right of dropdown
        y=1.15,
        xref="paper",
        yref="paper",
        text=subtitle_init,
        showarrow=False,
        align="left",
        font=dict(size=12, color="#2C3E50"),
    )
    subtitle_index = len(fig.layout.annotations) - 1  # index of the subtitle annotation

    for i, metric_name in enumerate(metric_order):
        vis = [True] * total_traces
        for j, trace_idx in enumerate(traces_panel_a):
            vis[trace_idx] = i == j
        vis[trace_p75] = False
        vis[trace_p25] = False

        subtitle_text = (
            f"<span style='font-size:12px; font-weight:bold;'>Metric:</span> {metric_label[metric_name]} | "
            f"<span style='font-size:12px; font-weight:bold;'>Normalize:</span> {normalize} | "
            f"<span style='font-size:12px; font-weight:bold;'>Min days:</span> {min_days}"
        )

        metric_buttons.append(
            dict(
                label=metric_label[metric_name],
                method="update",
                args=[
                    {"visible": vis},
                    {"annotations[{}].text".format(subtitle_index): subtitle_text},
                ],
            )
        )

    vis_iqr = [True] * total_traces
    for trace_idx in traces_panel_a:
        vis_iqr[trace_idx] = False
    vis_iqr[trace_p75] = True
    vis_iqr[trace_p25] = True

    metric_buttons.append(
        dict(
            label="IQR (25-75%)",
            method="update",
            args=[
                {"visible": vis_iqr},
                {"annotations[{}].text".format(subtitle_index): (
                    f"<span style='font-size:12px'>Metric:</span> IQR (spread) | <span style='font-size:12px'>Normalize:</span> {normalize} | <span style='font-size:12px'>Min sample size:</span> {min_days} days"
                )},
            ],
        )
    )

    fig.update_xaxes(title_text=x_title_panel_a, row=1, col=1, title_font=dict(size=12))
    fig.update_yaxes(title_text="Police District", row=1, col=1, title_font=dict(size=12))

    fig.update_xaxes(title_text="Top 3 districts by median", row=1, col=2, title_font=dict(size=12))
    fig.update_yaxes(title_text="Incidents per day", row=1, col=2, title_font=dict(size=12))

    fig.update_xaxes(title_text="(Mean - Median) / Median (%)", row=1, col=3, title_font=dict(size=12))
    fig.update_yaxes(title_text="Police District", row=1, col=3, title_font=dict(size=12))

    fig.add_annotation(
        x=0.2,
        y=0.05,
        xref="paper",
        yref="paper",
        text=(
            "<b>Color (ratio to city avg):</b><br>"
            "<span style='color:#E74C3C'>■</span> <i>> 1.2 (≥20% above)</i><br>"
            "<span style='color:#95A5A6'>■</span> <i>0.8 – 1.2 (≈ average)</i><br>"
            "<span style='color:#3498DB'>■</span> <i>< 0.8 (≥20% below)</i>"
        ),
        showarrow=False,
        align="left",
        font=dict(size=12, color="#2C3E50", family="Arial"),
    )

    fig.update_layout(
        updatemenus=[
            dict(
                buttons=metric_buttons,
                direction="down",
                showactive=True,
                x=0,
                y=1.2,
                # move dropdown to the right of the left-aligned subtitle
                xanchor="left",
                yanchor="top",
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="rgba(0,0,0,0.2)",
                borderwidth=1,
                font=dict(size=12),
                pad={"r": 8, "t": 8},
            )
        ],
        template="plotly_white",
        showlegend=False,
        height=780,
        hovermode="closest",
        margin=dict(l=100, r=90, t=150, b=90),
        bargap=0.18,
        legend=dict(
            orientation="h",
            y=-0.12,
            x=0.5,
            xanchor="center",
            yanchor="top",
            font=dict(size=11),
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="rgba(0,0,0,0.12)",
            borderwidth=1,
        ),
        hoverlabel=dict(font_size=11, font_family="Arial", bgcolor="white"),
        width=1700,

    )

    fig.update_annotations(font=dict(size=13, family="Arial"), yshift=8)
    fig.update_xaxes(tickfont=dict(size=10))
    fig.update_yaxes(tickfont=dict(size=10))

    return fig


fig = create_section_c_plot(year=None, categories=focus_crimes, normalize="ratio_to_city", min_days=10)
fig.show()

# fig.write_html("section_c_interactive.html", config={"responsive": True})

In [52]:
# Section C exportable HTML (pure Plotly controls with frames + slider + dropdown)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- Data prep ----------
_df = _get_section_c_df_v2()
_focus = _get_focus_crimes_v2(_df)
_df = _df[_df['Category'].isin(_focus)].copy()
_df = _df[_df['Year'] != 2026].copy()
_df = _df[_df['Police District'] != 'Out Of Sf'].copy()

if _df.empty:
    raise RuntimeError('No data available after filtering focus crimes and district.')

min_days = 10

daily = _df.groupby(['Year', 'Police District', 'Day']).size().reset_index(name='count')
sample_counts = daily.groupby(['Year', 'Police District'])['Day'].nunique().reset_index(name='n_days')
valid = sample_counts[sample_counts['n_days'] >= min_days][['Year', 'Police District']]
daily = daily.merge(valid, on=['Year', 'Police District'], how='inner')

if daily.empty:
    raise RuntimeError(f'No data available after applying min_days={min_days}.')

summary_rows = []
for (yr, district), grp in daily.groupby(['Year', 'Police District']):
    arr = grp['count'].to_numpy()
    med = float(np.median(arr))
    mean = float(np.mean(arr))
    tmean = float(_trimmed_mean_v2(arr, 0.1))
    p25 = float(np.percentile(arr, 25))
    p75 = float(np.percentile(arr, 75))
    diff_pct = ((mean - med) / med * 100.0) if med != 0 else 0.0
    summary_rows.append({
        'Year': int(yr),
        'Police District': district,
        'median': med,
        'mean': mean,
        'trimmed_mean': tmean,
        'p25': p25,
        'p75': p75,
        'diff_pct': float(diff_pct),
    })

sumdf = pd.DataFrame(summary_rows)
if sumdf.empty:
    raise RuntimeError('No summary rows could be computed.')

years = sorted(sumdf['Year'].dropna().astype(int).unique().tolist())
initial_year = years[-1]

# Helper for Panel 1 values per metric
metric_defs = {
    'median': {'label': 'Median', 'color': '#1E5BA8'},
    'mean': {'label': 'Mean', 'color': '#2874A6'},
    'trimmed_mean': {'label': 'Trimmed Mean (10%)', 'color': '#117A65'},
    'iqr_p75': {'label': '75th percentile', 'color': '#1E5BA8'},
    'iqr_p25': {'label': '25th percentile', 'color': '#AED6F1'},
}

# Precompute yearly trace payloads
year_payload = {}
for yr in years:
    ydf = sumdf[sumdf['Year'] == yr].copy()

    # Central metric order controls district order in panel 1 and 2
    med_ref = ydf['median'].mean() if not ydf.empty else 1.0
    med_ref = med_ref if med_ref != 0 else 1.0

    p1 = {}
    for m in ['median', 'mean', 'trimmed_mean']:
        ref = ydf[m].mean() if not ydf.empty else 1.0
        ref = ref if ref != 0 else 1.0
        temp = ydf[['Police District', m]].copy().rename(columns={m: 'value'})
        temp['ratio'] = temp['value'] / ref
        temp = temp.sort_values('ratio')
        temp['color'] = np.where(temp['ratio'] > 1.2, '#E74C3C', np.where(temp['ratio'] < 0.8, '#3498DB', '#95A5A6'))
        p1[m] = {
            'x': temp['ratio'].tolist(),
            'y': temp['Police District'].tolist(),
            'color': temp['color'].tolist(),
        }

    # IQR traces use median ordering for visual consistency
    base_order = p1['median']['y']
    temp_iqr = ydf.set_index('Police District').reindex(base_order)
    p75_ratio = (temp_iqr['p75'] / med_ref).fillna(0).tolist()
    p25_ratio = (temp_iqr['p25'] / med_ref).fillna(0).tolist()
    p1['iqr_p75'] = {'x': p75_ratio, 'y': base_order, 'color': [metric_defs['iqr_p75']['color']] * len(base_order)}
    p1['iqr_p25'] = {'x': p25_ratio, 'y': base_order, 'color': [metric_defs['iqr_p25']['color']] * len(base_order)}

    # Panel 2 (all daily points for that year)
    d2 = daily[daily['Year'] == yr].copy().sort_values(['Police District', 'Day'])
    p2_x = d2['Police District'].tolist()
    p2_y = d2['count'].tolist()

    # Panel 3 (sorted by skew)
    d3 = ydf[['Police District', 'diff_pct']].copy().sort_values('diff_pct')
    p3_x = d3['diff_pct'].tolist()
    p3_y = d3['Police District'].tolist()
    p3_c = ['#EF9343' if v > 0 else '#1ABC9C' for v in p3_x]

    year_payload[yr] = {
        'p1': p1,
        'p2': {'x': p2_x, 'y': p2_y},
        'p3': {'x': p3_x, 'y': p3_y, 'color': p3_c},
    }

# ---------- Build figure ----------
fig = make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.34, 0.34, 0.32],
    horizontal_spacing=0.09,
    subplot_titles=(
        "<span style='font-size:18px;'><b>Panel 1.</b> District Comparison Depends on Summary Metric</span>",
        "<span style='font-size:18px;'><b>Panel 2.</b> Distribution Reveals Daily Volatility</span>",
        "<span style='font-size:18px;'><b>Panel 3.</b> Mean-Median Gap Flags Skew</span>",
    ),
)

init = year_payload[initial_year]

# Trace 0: median (visible)
fig.add_trace(go.Bar(
    x=init['p1']['median']['x'], y=init['p1']['median']['y'], orientation='h',
    marker=dict(color=init['p1']['median']['color']),
    opacity=0.75, name='Median', showlegend=False,
    hovertemplate='%{y}<br>Ratio to city average: %{x:.2f}<extra></extra>'
), row=1, col=1)

# Trace 1: mean (hidden)
fig.add_trace(go.Bar(
    x=init['p1']['mean']['x'], y=init['p1']['mean']['y'], orientation='h',
    marker=dict(color=init['p1']['mean']['color']),
    opacity=0.75, name='Mean', showlegend=False, visible=False,
    hovertemplate='%{y}<br>Ratio to city average: %{x:.2f}<extra></extra>'
), row=1, col=1)

# Trace 2: trimmed mean (hidden)
fig.add_trace(go.Bar(
    x=init['p1']['trimmed_mean']['x'], y=init['p1']['trimmed_mean']['y'], orientation='h',
    marker=dict(color=init['p1']['trimmed_mean']['color']),
    opacity=0.75, name='Trimmed Mean (10%)', showlegend=False, visible=False,
    hovertemplate='%{y}<br>Ratio to city average: %{x:.2f}<extra></extra>'
), row=1, col=1)

# Trace 3: IQR p75 (hidden)
fig.add_trace(go.Bar(
    x=init['p1']['iqr_p75']['x'], y=init['p1']['iqr_p75']['y'], orientation='h',
    marker=dict(color=init['p1']['iqr_p75']['color']),
    opacity=0.5, name='75th percentile', showlegend=False, visible=False,
    hovertemplate='%{y}<br>75th percentile ratio: %{x:.2f}<extra></extra>'
), row=1, col=1)

# Trace 4: IQR p25 (hidden)
fig.add_trace(go.Bar(
    x=init['p1']['iqr_p25']['x'], y=init['p1']['iqr_p25']['y'], orientation='h',
    marker=dict(color=init['p1']['iqr_p25']['color']),
    opacity=0.5, name='25th percentile', showlegend=False, visible=False,
    hovertemplate='%{y}<br>25th percentile ratio: %{x:.2f}<extra></extra>'
), row=1, col=1)

# Trace 5: panel 2 violin (visible)
fig.add_trace(go.Violin(
    x=init['p2']['x'], y=init['p2']['y'],
    box_visible=False, meanline_visible=True, points=False,
    line_color='#2E8B57', fillcolor='rgba(46,139,87,0.38)', opacity=0.65,
    showlegend=False,
    hovertemplate='District: %{x}<br>Incidents/day: %{y}<extra></extra>'
), row=1, col=2)

# Trace 6: panel 3 skew bar (visible)
fig.add_trace(go.Bar(
    x=init['p3']['x'], y=init['p3']['y'], orientation='h',
    marker=dict(color=init['p3']['color']), opacity=0.82,
    showlegend=False,
    hovertemplate='%{y}<br>(Mean - Median) / Median: %{x:.1f}%<extra></extra>'
), row=1, col=3)

fig.add_vline(x=1.0, line_width=2, line_dash='dash', line_color='#2C3E50', row=1, col=1)
fig.add_vline(x=0, line_width=2, line_dash='dot', line_color='#7F8C8D', row=1, col=3)

fig.update_xaxes(title_text='Ratio to city average (=1.0)', row=1, col=1, title_font=dict(size=12))
fig.update_yaxes(title_text='Police District', row=1, col=1, title_font=dict(size=12))

fig.update_xaxes(title_text='Police District', row=1, col=2, title_font=dict(size=12), tickangle=-35)
fig.update_yaxes(title_text='Incidents per day', row=1, col=2, title_font=dict(size=12))

fig.update_xaxes(title_text='(Mean - Median) / Median (%)', row=1, col=3, title_font=dict(size=12))
fig.update_yaxes(title_text='Police District', row=1, col=3, title_font=dict(size=12))

# Dropdown for panel 1 metric visibility only
metric_buttons = [
    dict(label='Median', method='update', args=[{'visible': [True, False, False, False, False, True, True]}]),
    dict(label='Mean', method='update', args=[{'visible': [False, True, False, False, False, True, True]}]),
    dict(label='Trimmed Mean (10%)', method='update', args=[{'visible': [False, False, True, False, False, True, True]}]),
    dict(label='IQR (25th-75th)', method='update', args=[{'visible': [False, False, False, True, True, True, True]}]),
]

# Frames for year slider update all traces' data
frames = []
for yr in years:
    p = year_payload[yr]
    frames.append(go.Frame(
        name=str(yr),
        data=[
            go.Bar(x=p['p1']['median']['x'], y=p['p1']['median']['y'], marker=dict(color=p['p1']['median']['color'])),
            go.Bar(x=p['p1']['mean']['x'], y=p['p1']['mean']['y'], marker=dict(color=p['p1']['mean']['color'])),
            go.Bar(x=p['p1']['trimmed_mean']['x'], y=p['p1']['trimmed_mean']['y'], marker=dict(color=p['p1']['trimmed_mean']['color'])),
            go.Bar(x=p['p1']['iqr_p75']['x'], y=p['p1']['iqr_p75']['y'], marker=dict(color=p['p1']['iqr_p75']['color'])),
            go.Bar(x=p['p1']['iqr_p25']['x'], y=p['p1']['iqr_p25']['y'], marker=dict(color=p['p1']['iqr_p25']['color'])),
            go.Violin(x=p['p2']['x'], y=p['p2']['y'], box_visible=False, meanline_visible=True, points=False),
            go.Bar(x=p['p3']['x'], y=p['p3']['y'], marker=dict(color=p['p3']['color'])),
        ]
    ))

fig.frames = frames

slider_steps = [
    dict(
        method='animate',
        args=[[str(yr)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
        label=str(yr),
    )
    for yr in years
]

fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=840,
    width=1700,
    hovermode='closest',
    margin=dict(l=100, r=90, t=165, b=115),
    bargap=0.18,
    hoverlabel=dict(font_size=11, font_family='Arial', bgcolor='white'),
    # title=f"Section C (Plotly-native controls) | Year: {initial_year}",
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            buttons=metric_buttons,
            showactive=True,
            x=0.0,
            y=1.20,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.92)',
            bordercolor='rgba(0,0,0,0.2)',
            borderwidth=1,
            font=dict(size=12),
            pad={'r': 8, 't': 8},
        )
    ],
    sliders=[
        dict(
            active=len(years) - 1,
            currentvalue={'prefix': 'Year: ', 'font': {'size': 12}},
            pad={'t': 40},
            len=0.55,
            x=0.25,
            y=1.4,
            xanchor='left',
            steps=slider_steps,
        )
    ],
)

fig.add_annotation(
    x=0.18,
    y=0.04,
    xref='paper',
    yref='paper',
    text=(
        "<b>Panel 1 color (ratio to city avg):</b><br>"
        "<span style='color:#E74C3C'>■</span> > 1.2 (above city avg)<br>"
        "<span style='color:#95A5A6'>■</span> 0.8 - 1.2 (near avg)<br>"
        "<span style='color:#3498DB'>■</span> < 0.8 (below avg)"
    ),
    showarrow=False,
    align='left',
    font=dict(size=12, color='#2C3E50', family='Arial'),
    width=1700,
)

fig.show()
fig.write_html('section_c_interactive_embeddable.html', include_plotlyjs='cdn', full_html=True, config={'responsive': True})
print('Wrote section_c_interactive_embeddable.html (Plotly-native slider + dropdown).')

Wrote section_c_interactive_embeddable.html (Plotly-native slider + dropdown).
